# Eight-arm SLM harness — M1 GPU smoke test

Runs on **Kaggle, Accelerator GPU T4 (x1 or x2), Internet ON**. Optional secret `GITHUB_TOKEN`
(push access) to push the results back to the branch; without it, results stay in `/kaggle/working/m1_smoke/`.

What it does (Run All, ~40–50 min wall, ~25 min of GPU inference):
1. clones branch `harness/eight-arm-slm`, installs the harness;
2. starts Ollama on GPU 0 with `qwen2.5-coder:14b-instruct-q4_K_M`, 16,384-token window, flash attention, q8_0 KV cache;
3. **CPU smoke test** (`harness.smoke_test_cpu`): the real jina embedder and the bge ONNX reranker load here (they were blocked in the dev container);
4. **M1 gate** (`harness.kaggle_m1`): tokenizer parity vs the serving model, then
   Gate A — 5 synthetic tasks (T1–T5) × Arms 0/1/5/Oracle, Gate B — 5 real FastAPI T2 tasks × the same arms;
5. prints PASS/FAIL per (arm, task_type) and saves everything to `/kaggle/working/m1_smoke/`.

Embedders and the reranker run on **CPU only** (`CUDA_VISIBLE_DEVICES=""` for the harness process); the GPU belongs to Ollama.

In [ ]:
# ---- settings ----
BRANCH = "harness/eight-arm-slm"
REPO_URL = "https://github.com/rohith-hash-stack/SCE.git"
MODEL = "qwen2.5-coder:14b-instruct-q4_K_M"
NUM_CTX = 16384
OLLAMA_VERSION = "0.34.0"      # same pin as scripts/kaggle_slm_sweep.py; "" = latest
SEED = 42
OUT = "/kaggle/working/m1_smoke"
SCE_DIR = "/kaggle/working/SCE"
PUSH_RESULTS = True            # needs the GITHUB_TOKEN secret

In [ ]:
import json, os, subprocess, sys, time, urllib.request

def run(cmd, cwd=None, check=True, env=None, capture=False):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    tok = os.environ.get("GITHUB_TOKEN")
    print("$", shown.replace(tok, "***") if tok else shown, flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check, env=env, shell=isinstance(cmd, str), capture_output=capture, text=True)

def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
if token:
    os.environ["GITHUB_TOKEN"] = token
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
if not os.path.isdir(SCE_DIR):
    run(["git", "clone", "--depth", "50", "--branch", BRANCH, url, SCE_DIR])
else:
    run(["git", "fetch", "origin", BRANCH], cwd=SCE_DIR); run(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], cwd=SCE_DIR)
print(run(["git", "log", "--oneline", "-1"], cwd=SCE_DIR, capture=True).stdout)

In [ ]:
# ---- install (CPU-side packages; torch is preinstalled on Kaggle) ----
run(f"{sys.executable} -m pip install -q -e {SCE_DIR}")
run(f"{sys.executable} -m pip install -q rank-bm25 sentence-transformers onnxruntime onnx onnxscript einops "
    "pandas pyarrow statsmodels pydantic tree-sitter tree-sitter-python")
run("apt-get -y -qq install ripgrep zstd > /dev/null", check=False)
run(["rg", "--version"])

In [ ]:
# ---- Ollama on GPU 0 only ----
gpus = [g for g in run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture=True).stdout.splitlines() if g.strip()]
assert gpus, "no GPU: set Accelerator to GPU T4"
print("GPUs:", gpus)
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    pin = f"OLLAMA_VERSION={OLLAMA_VERSION} " if OLLAMA_VERSION else ""
    run(f"curl -fsSL https://ollama.com/install.sh | {pin}sh")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(NUM_CTX),
           OLLAMA_NUM_PARALLEL="1", OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0",
           OLLAMA_KEEP_ALIVE="-1", OLLAMA_MODELS="/kaggle/working/ollama-models")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=5); break
    except Exception:
        time.sleep(2)
run(["ollama", "pull", MODEL], env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434"))
# warm-up: load the model into VRAM once
req = urllib.request.Request("http://127.0.0.1:11434/api/generate", data=json.dumps(
    {"model": MODEL, "prompt": "ok", "stream": False, "options": {"num_predict": 1}}).encode(), headers={"Content-Type": "application/json"})
urllib.request.urlopen(req, timeout=900).read()
print(run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv"], capture=True).stdout)

In [ ]:
# ---- CPU smoke test (real jina + bge ONNX on CPU) ----
harness_env = dict(os.environ, CUDA_VISIBLE_DEVICES="", PYTHONPATH=SCE_DIR, HARNESS_WORKDIR="/kaggle/working",
                   LLM_BASE_URL="http://127.0.0.1:11434/v1")
os.makedirs(OUT, exist_ok=True)
smoke = run([sys.executable, "-m", "harness.smoke_test_cpu", "--json", f"{OUT}/smoke_cpu.json"], cwd=SCE_DIR, env=harness_env, check=False)
print("CPU smoke exit code:", smoke.returncode)

In [ ]:
# ---- M1 gate: tokenizer parity, Gate A (type coverage), Gate B (5 real FastAPI T2) ----
t0 = time.time()
gate = run([sys.executable, "-m", "harness.kaggle_m1", "--out", OUT, "--seed", str(SEED), "--model", MODEL,
            "--llm-url", "http://127.0.0.1:11434/v1", "--ollama-url", "http://127.0.0.1:11434"],
           cwd=SCE_DIR, env=harness_env, check=False)
print(f"M1 gate exit code: {gate.returncode}  ({(time.time() - t0) / 60:.1f} min)")

In [ ]:
# ---- summary + optional push ----
rep = json.load(open(f"{OUT}/gate_report.json"))
sm = json.load(open(f"{OUT}/smoke_cpu.json"))
print("CPU smoke:", sm["counts"])
print("tokenizer parity:", rep["tokenizer_parity"].get("status"), rep["tokenizer_parity"].get("max_divergence"))
print("rows:", sum(r["status"] == "PASS" for r in rep["rows"]), "PASS /", len(rep["rows"]))
if PUSH_RESULTS and os.environ.get("GITHUB_TOKEN"):
    dest = f"{SCE_DIR}/reports/harness_m1/kaggle_smoke"
    run(f"rm -rf {dest} && mkdir -p {dest} && cp -r {OUT}/. {dest}/")
    run(["git", "config", "user.email", "kaggle-runner@users.noreply.github.com"], cwd=SCE_DIR)
    run(["git", "config", "user.name", "kaggle-runner"], cwd=SCE_DIR)
    run(["git", "add", "-f", "reports/harness_m1/kaggle_smoke"], cwd=SCE_DIR)
    run(["git", "commit", "-m", f"M1 Kaggle smoke results ({MODEL}, seed {SEED})"], cwd=SCE_DIR, check=False)
    for attempt in range(4):
        if run(["git", "push", "origin", f"HEAD:{BRANCH}"], cwd=SCE_DIR, check=False).returncode == 0:
            break
        run(["git", "pull", "--rebase", "origin", BRANCH], cwd=SCE_DIR, check=False); time.sleep(2 ** (attempt + 1))